# Detection DF Scaling Study

This notebook scales the original experiment to a medium-size subset of the dataset.

Protocol used here:
- split the dataset into `ai` and `real`
- keep only the first 25 videos from `ai` and the first 25 videos from `real`
- for each video, extract the first 6 frames
- evaluate 5 consecutive pairs: `0->1`, `1->2`, `2->3`, `3->4`, `4->5`
- compute `mean_squared_error` and `valid_pixel_ratio` for each pair
- average these metrics at the video level
- average the video-level metrics over all fake videos, then over all real videos
- save every pair result, every video summary, and both dataset summaries into one CSV file

This keeps the aggregation rule aligned with the goal: compare fake vs real at a larger scale.


In [10]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import sys

import numpy as np


In [11]:
ROOT = Path.cwd()
if not (ROOT / "video_pipeline.py").exists() and (ROOT / "python_visual_odometry" / "video_pipeline.py").exists():
    ROOT = ROOT / "python_visual_odometry"

PROJECT_ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import video_pipeline as vp

DATASET_ROOT = PROJECT_ROOT / "dataset"
AI_DIR = DATASET_ROOT / "ai"
REAL_DIR = DATASET_ROOT / "real"
OUTPUT_DIR = ROOT / "scaling_outputs"
FRAME_DIR = OUTPUT_DIR / "frames"
CSV_PATH = OUTPUT_DIR / "detection_df_scaling_stats.csv"

FRAME_COUNT = 6
MAX_VIDEOS_PER_SPLIT = 25
START_FRAME = 0
MIN_GAP = 1
RESIZE_WIDTH = None
USE_CACHE = True
SAVE_CACHE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("DATASET_ROOT =", DATASET_ROOT)
print("AI_DIR =", AI_DIR)
print("REAL_DIR =", REAL_DIR)
print("CSV_PATH =", CSV_PATH)
print("FRAME_COUNT =", FRAME_COUNT)
print("MAX_VIDEOS_PER_SPLIT =", MAX_VIDEOS_PER_SPLIT)


ROOT = c:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\python_visual_odometry
DATASET_ROOT = c:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset
AI_DIR = c:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset\ai
REAL_DIR = c:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset\real
CSV_PATH = c:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\python_visual_odometry\scaling_outputs\detection_df_scaling_stats.csv
FRAME_COUNT = 6
MAX_VIDEOS_PER_SPLIT = 25


In [12]:
def list_video_files(directory: Path, max_videos: int | None = None) -> list[Path]:
    video_files = sorted(
        [path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}],
        key=lambda path: path.name.lower(),
    )
    if max_videos is not None:
        video_files = video_files[:max_videos]
    return video_files


def analyze_frame_pair(keyframe_path: Path, target_path: Path) -> dict:
    keyframe_image = vp.read_image_gray(keyframe_path)
    target_image = vp.read_image_gray(target_path)

    keyframe_depth = vp.infer_depth_map(keyframe_path, use_cache=USE_CACHE, save_cache=SAVE_CACHE)
    keyframe_inv_depth, keyframe_inv_depth_var, depth_valid_mask = vp.depth_to_inverse_depth(keyframe_depth)

    intrinsics = vp.estimate_intrinsics(keyframe_image.shape[1], keyframe_image.shape[0])
    cam = vp.camera.camera(
        intrinsics["fx"],
        intrinsics["fy"],
        intrinsics["cx"],
        intrinsics["cy"],
        intrinsics["width"],
        intrinsics["height"],
    )

    keyframe = vp.frameData.frameData()
    keyframe.setImage(keyframe_image)
    keyframe.setInvDepth(keyframe_inv_depth, keyframe_inv_depth_var)

    target_frame = vp.frameData.frameData()
    target_frame.setImage(target_image)

    pose_solver = vp.pose_estimator_gauss_newton.pose_estimator_gauss_newton(cam, show_debug=False)
    pose_solver.optPose(target_frame, keyframe)
    final_error_lvl2, _ = pose_solver.computeError(target_frame, keyframe, lvl=2)

    photometric_maps = vp.compute_photometric_maps(target_frame, keyframe, cam, lvl=0)
    squared_diff_map = photometric_maps["squared_diff_map"]
    valid_mask = photometric_maps["valid_mask"]

    valid_pixel_count = int(valid_mask.sum())
    total_pixel_count = int(valid_mask.size)
    if valid_pixel_count == 0:
        raise RuntimeError("No valid reprojected pixels for this frame pair.")

    return {
        "mean_squared_error": float(np.nanmean(squared_diff_map)),
        "valid_pixel_count": valid_pixel_count,
        "total_pixel_count": total_pixel_count,
        "valid_pixel_ratio": float(valid_pixel_count / total_pixel_count),
        "depth_valid_ratio": float(depth_valid_mask.mean()),
        "final_error_lvl2": float(final_error_lvl2),
        "keyframe_path": str(keyframe_path),
        "target_path": str(target_path),
    }


In [13]:
def analyze_video(video_path: Path, split_name: str) -> list[dict]:
    video_output_dir = FRAME_DIR / split_name / video_path.stem
    extraction = vp.extract_frames_from_video(
        video_path=video_path,
        output_dir=video_output_dir,
        frame_count=FRAME_COUNT,
        start_frame=START_FRAME,
        min_gap=MIN_GAP,
        resize_width=RESIZE_WIDTH,
    )

    rows: list[dict] = []
    for pair_index in range(len(extraction.frame_paths) - 1):
        keyframe_path = extraction.frame_paths[pair_index]
        target_path = extraction.frame_paths[pair_index + 1]
        pair_metrics = analyze_frame_pair(keyframe_path, target_path)
        rows.append(
            {
                "row_type": "pair",
                "split": split_name,
                "video_name": video_path.name,
                "video_path": str(video_path),
                "pair_index": int(pair_index),
                "frame_count_used": int(len(extraction.frame_paths)),
                "selected_video_indices": json.dumps(extraction.selected_indices),
                "keyframe_name": keyframe_path.name,
                "target_name": target_path.name,
                "keyframe_path": pair_metrics["keyframe_path"],
                "target_path": pair_metrics["target_path"],
                "mean_squared_error": pair_metrics["mean_squared_error"],
                "valid_pixel_count": pair_metrics["valid_pixel_count"],
                "total_pixel_count": pair_metrics["total_pixel_count"],
                "valid_pixel_ratio": pair_metrics["valid_pixel_ratio"],
                "depth_valid_ratio": pair_metrics["depth_valid_ratio"],
                "final_error_lvl2": pair_metrics["final_error_lvl2"],
                "status": "ok",
                "error_message": "",
            }
        )

    pair_rows = [row for row in rows if row["row_type"] == "pair"]
    rows.append(
        {
            "row_type": "video_summary",
            "split": split_name,
            "video_name": video_path.name,
            "video_path": str(video_path),
            "pair_index": "",
            "frame_count_used": int(len(extraction.frame_paths)),
            "selected_video_indices": json.dumps(extraction.selected_indices),
            "keyframe_name": "",
            "target_name": "",
            "keyframe_path": "",
            "target_path": "",
            "mean_squared_error": float(np.mean([row["mean_squared_error"] for row in pair_rows])),
            "valid_pixel_count": "",
            "total_pixel_count": "",
            "valid_pixel_ratio": float(np.mean([row["valid_pixel_ratio"] for row in pair_rows])),
            "depth_valid_ratio": float(np.mean([row["depth_valid_ratio"] for row in pair_rows])),
            "final_error_lvl2": float(np.mean([row["final_error_lvl2"] for row in pair_rows])),
            "status": "ok",
            "error_message": "",
        }
    )
    return rows


In [14]:
def safe_analyze_split(split_name: str, directory: Path) -> list[dict]:
    rows: list[dict] = []
    video_files = list_video_files(directory, max_videos=MAX_VIDEOS_PER_SPLIT)
    print(f"Processing {split_name}: {len(video_files)} videos")

    for index, video_path in enumerate(video_files, start=1):
        print(f"[{split_name}] {index}/{len(video_files)} -> {video_path.name}")
        try:
            video_rows = analyze_video(video_path, split_name)
            rows.extend(video_rows)
        except Exception as exc:
            rows.append(
                {
                    "row_type": "video_summary",
                    "split": split_name,
                    "video_name": video_path.name,
                    "video_path": str(video_path),
                    "pair_index": "",
                    "frame_count_used": "",
                    "selected_video_indices": "",
                    "keyframe_name": "",
                    "target_name": "",
                    "keyframe_path": "",
                    "target_path": "",
                    "mean_squared_error": "",
                    "valid_pixel_count": "",
                    "total_pixel_count": "",
                    "valid_pixel_ratio": "",
                    "depth_valid_ratio": "",
                    "final_error_lvl2": "",
                    "status": "error",
                    "error_message": str(exc),
                }
            )
            print(f"  ERROR: {exc}")
    return rows


In [15]:
all_rows = []
all_rows.extend(safe_analyze_split("ai", AI_DIR))
all_rows.extend(safe_analyze_split("real", REAL_DIR))

video_summary_rows = [
    row for row in all_rows
    if row["row_type"] == "video_summary" and row["status"] == "ok"
]

for split_name in ["ai", "real"]:
    split_video_rows = [row for row in video_summary_rows if row["split"] == split_name]
    if not split_video_rows:
        continue
    all_rows.append(
        {
            "row_type": "dataset_summary",
            "split": split_name,
            "video_name": "ALL",
            "video_path": "",
            "pair_index": "",
            "frame_count_used": "",
            "selected_video_indices": "",
            "keyframe_name": "",
            "target_name": "",
            "keyframe_path": "",
            "target_path": "",
            "mean_squared_error": float(np.mean([row["mean_squared_error"] for row in split_video_rows])),
            "valid_pixel_count": "",
            "total_pixel_count": "",
            "valid_pixel_ratio": float(np.mean([row["valid_pixel_ratio"] for row in split_video_rows])),
            "depth_valid_ratio": float(np.mean([row["depth_valid_ratio"] for row in split_video_rows])),
            "final_error_lvl2": float(np.mean([row["final_error_lvl2"] for row in split_video_rows])),
            "status": "ok",
            "error_message": "",
        }
    )

fieldnames = [
    "row_type",
    "split",
    "video_name",
    "video_path",
    "pair_index",
    "frame_count_used",
    "selected_video_indices",
    "keyframe_name",
    "target_name",
    "keyframe_path",
    "target_path",
    "mean_squared_error",
    "valid_pixel_count",
    "total_pixel_count",
    "valid_pixel_ratio",
    "depth_valid_ratio",
    "final_error_lvl2",
    "status",
    "error_message",
]

with CSV_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"Saved CSV to: {CSV_PATH}")
print(f"Total rows written: {len(all_rows)}")


Processing ai: 25 videos
[ai] 1/25 -> 19.mp4
lvl:  4  initial error:  164.19372900335932
 error improvement too small, level converged! it:  4  error:  156.57588077711767  lambda:  0.2
lvl:  3  initial error:  153.95951142236243
 error improvement too small, level converged! it:  4  error:  143.60899434225533  lambda:  0.0
lvl:  2  initial error:  161.21924775228
 error improvement too small, level converged! it:  2  error:  158.25749848049847  lambda:  0.0
lvl:  4  initial error:  188.66033519553054
update too small, level converged! it:  6  error:  183.46952079430517  lambda:  3518437208883.2
lvl:  3  initial error:  193.17869773380363
 error improvement too small, level converged! it:  2  error:  179.15562438953322  lambda:  109951162777.6
lvl:  2  initial error:  172.89642642305128
 error improvement too small, level converged! it:  3  error:  171.81392672952285  lambda:  0.0
lvl:  4  initial error:  165.69887640449417
 error improvement too small, level converged! it:  3  error:  

In [16]:
dataset_summary_rows = [row for row in all_rows if row["row_type"] == "dataset_summary"]
video_ok_count = len([row for row in all_rows if row["row_type"] == "video_summary" and row["status"] == "ok"])
video_error_count = len([row for row in all_rows if row["row_type"] == "video_summary" and row["status"] == "error"])

print("Successful video summaries =", video_ok_count)
print("Failed video summaries     =", video_error_count)
print()
for row in dataset_summary_rows:
    print(row["split"].upper())
    print("  mean video MSE         =", row["mean_squared_error"])
    print("  mean valid pixel ratio =", row["valid_pixel_ratio"])
    print("  mean depth valid ratio =", row["depth_valid_ratio"])
    print("  mean final error lvl2  =", row["final_error_lvl2"])
    print()


Successful video summaries = 50
Failed video summaries     = 0

AI
  mean video MSE         = 192.3626810750961
  mean valid pixel ratio = 0.8668890075683593
  mean depth valid ratio = 0.8763888471408556
  mean final error lvl2  = 225.95759048306343

REAL
  mean video MSE         = 218.1079581003189
  mean valid pixel ratio = 0.8644171752929688
  mean depth valid ratio = 0.8733612648222215
  mean final error lvl2  = 222.69402603795285



The results are inconclusive...